## tl;dr

本 Notebook 审计 2026-08-21 至 2026-08-23 的 Hilo、Plinko 生命周期聚合数据。Plinko 的 GM 真实回报比偏离预期较大，是优先核查信号；当前没有用户、局、机器人和配置命中字段，因此不认定羊毛或机器人。

## Context & Methods

- 数据源：`data/outputs/lifecycle_joint/2026-08-24-lark-update/source-data.json`。
- 粒度：`date × game`、`date × lifecycle × game`。
- 口径：GM 真实回报比 = `1 - 实际盈利 / 下注额`；不等同订单收入 LTV 或独立实测 RTP。
- 计算：三日指标由累计分子/分母重算，不平均每日比例。
- 样本：新游戏只有三个完整自然日，D1/D3/D7 不作结论。

## Data

读取已通过表头、跨表勾稽和日期完整性校验的源快照，并调用同一分析模块生成结果，确保 Notebook 与 HTML 报告使用相同计算路径。

In [1]:
from pathlib import Path
import importlib.util
import json

ROOT = Path.cwd()
source_path = ROOT / 'data/outputs/lifecycle_joint/2026-08-24-lark-update/source-data.json'
script_path = ROOT / 'scripts/build_new_games_lifecycle_wool_audit.py'
spec = importlib.util.spec_from_file_location('new_games_audit', script_path)
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
source = json.loads(source_path.read_text(encoding='utf-8'))
analysis = audit.build_analysis(source)
analysis['headline']

{'observation_days': 3,
 'hilo_base_bet': 35400.0,
 'plinko_base_bet': 791298.05,
 'plinko_gap_pp': 13.261,
 'hilo_actual_return_ratio': 0.864534,
 'plinko_actual_return_ratio': 1.101657}

## Results

先验证新游戏首次出现日期、观察窗口和关键聚合指标，再检查生命周期 0–4 集中度和风险状态。

In [2]:
daily = analysis['datasets']['daily_game_metrics']
summary = analysis['datasets']['game_summary']
mix = analysis['datasets']['lifecycle_mix']
assert sorted({row['date'] for row in daily if row['game'] in {'Hilo', 'Plinko'}}) == ['2026-08-21', '2026-08-22', '2026-08-23']
assert analysis['quality']['long_retention'] == 'not_mature'
for row in summary:
    if row['game'] in {'Hilo', 'Plinko'}:
        print(row)
print('lifecycle4 shares:', {g: next(x['base_bet_share'] for x in mix if x['game']==g and x['lifecycle']==4) for g in ['Hilo','Plinko']})

{'game': 'Plinko', 'observed_days': 3, 'base_bet': 791298.05, 'entire_bet': 804158.05, 'base_actual_profit': -80441.33, 'base_expected_profit': 24496.05, 'entire_actual_profit': -77802.33, 'entire_expected_profit': 24881.85, 'actual_return_ratio': 1.101657, 'expected_return_ratio': 0.969043, 'entire_actual_return_ratio': 1.09675, 'entire_expected_return_ratio': 0.969059, 'return_gap_pp': 13.261, 'entire_return_gap_pp': 12.769, 'lifecycle4_share': 0.604757}
{'game': 'Hilo', 'observed_days': 3, 'base_bet': 35400.0, 'entire_bet': 35400.0, 'base_actual_profit': 4795.5, 'base_expected_profit': 1068.0, 'entire_actual_profit': 4795.5, 'entire_expected_profit': 1068.0, 'actual_return_ratio': 0.864534, 'expected_return_ratio': 0.969831, 'entire_actual_return_ratio': 0.864534, 'entire_expected_return_ratio': 0.969831, 'return_gap_pp': -10.53, 'entire_return_gap_pp': -10.53, 'lifecycle4_share': 0.631356}
lifecycle4 shares: {'Hilo': 0.631356, 'Plinko': 0.604757}


## Takeaways

Plinko 的偏差应优先核对有效下注、最终派奖、免费注、跨日结算和配置版本；Hilo 需要扩大样本后再判断。两款游戏的羊毛判断需要补齐 `user_id_hash`、`round_id`、`is_robot`、`config_version` 和 `settlement_status`。